In [ ]:
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client
from strands import Agent
from strands.tools.mcp import MCPClient
from strands.models import BedrockModel

GATEWAY_URL = "https://gateway-ehjdbglfzt.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
GATEWAY_URL = "https://gateway-for-amazon-quick-wautpoplwm.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"

mcp_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_service="bedrock-agentcore",
    ),
)

# List available tools first
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print("Available tools:")
    for t in tools:
        print(f"  - {t.tool_name}")

In [ ]:
model_id = "global.anthropic.claude-opus-4-6-v1"
model = BedrockModel(
    model_id=model_id,
)

# Create agent and use the tools
agent = Agent(model=model, tools=[mcp_client])
response = agent("weather in athens please")
print(f"\nAgent response: {response}")

In [ ]:
# Create agent and use the tools
agent = Agent(model=model, tools=[mcp_client])
response = agent("I have a budget of $50. Find me the best value football bet right now — check the events, compare the odds, pick the one with the highest potential return, and place the bet for me. Then show me my updated balance.")
print(f"\nAgent response: {response}")

In [ ]:
response = agent("show me details of this last bet")

## Gateway Semantic Tool Search

AgentCore Gateway can index all tool metadata into a managed vector store and expose a built-in
`x_amz_bedrock_agentcore_search` MCP tool. You search with natural language, get back only the
relevant tools, and hand those to the agent — keeping the context window small and latency low.

> **Prerequisite:** The gateway must be created with `protocolConfiguration = {"mcp": {"searchType": "SEMANTIC"}}`.
> This cannot be added after creation.

In [ ]:
import json
from strands.tools.mcp.mcp_client import MCPClient, MCPAgentTool
from mcp.types import Tool as MCPTool

# Point to a gateway that has semantic search enabled
# GATEWAY_URL = "https://your-search-enabled-gateway.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"

search_client = MCPClient(
    lambda: aws_iam_streamablehttp_client(
        endpoint=GATEWAY_URL,
        aws_service="bedrock-agentcore",
    ),
)

### Step 1 — Search for tools with a natural language query

In [ ]:
with search_client:
    result = search_client.call_tool_sync(
        tool_use_id="search-1",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "place a bet on a football match and check odds"},
    )
    # The search returns tool definitions ranked by relevance
    found_tools = json.loads(result["content"][0]["text"])["tools"]
    print(f"Found {len(found_tools)} relevant tools:")
    for t in found_tools:
        print(f"  - {t['name']}: {t['description'][:80]}...")

### Step 2 — Convert search results into agent tools and run the agent

In [ ]:
def search_to_agent_tools(search_results, client, top_n=5):
    """Convert gateway search results into Strands MCPAgentTool objects."""
    agent_tools = []
    for tool in search_results[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        agent_tools.append(MCPAgentTool(mcp_tool, client))
    return agent_tools


with search_client:
    # Search for betting tools
    result = search_client.call_tool_sync(
        tool_use_id="search-2",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "place a bet, check odds, view balance"},
    )
    found_tools = json.loads(result["content"][0]["text"])["tools"]

    # Build a focused agent with only the relevant tools
    focused_agent = Agent(
        model=model,
        tools=search_to_agent_tools(found_tools, search_client, top_n=5),
        system_prompt="You are a betting assistant for a sports betting platform.",
    )
    response = focused_agent("What are the odds for the Arsenal vs Chelsea match?")
    print(response.message["content"][0]["text"])

### Step 3 — Different query, different tools, same gateway

In [ ]:
with search_client:
    # Now search for customer support tools instead
    result = search_client.call_tool_sync(
        tool_use_id="search-3",
        name="x_amz_bedrock_agentcore_search",
        arguments={"query": "customer account details, transactions, support tickets"},
    )
    support_tools = json.loads(result["content"][0]["text"])["tools"]
    print(f"Found {len(support_tools)} support tools:")
    for t in support_tools:
        print(f"  - {t['name']}")

    # Build a support agent with only those tools
    support_agent = Agent(
        model=model,
        tools=search_to_agent_tools(support_tools, search_client, top_n=5),
        system_prompt="You are a customer support agent for a sports betting platform.",
    )
    response = support_agent("Show me the transaction history for customer cust-003")
    print(response.message["content"][0]["text"])